In [1]:
#Importing Libraries
from keras.layers import Input,Dense,Flatten
from keras.models import Model
from tensorflow.keras import layers
from keras.applications.xception import Xception
from keras.applications.xception import preprocess_input
from keras.preprocessing import image
from tensorflow.keras.utils import image_dataset_from_directory #replace with keras.utils.image_dataset_from_directory as per the latest keras version
from keras.models import Sequential
import numpy as np
from glob import glob

In [2]:
from google.colab import files
uploaded = files.upload()  # pick your kaggle.json

import os, shutil

# Automatically find whatever kaggle json file was just uploaded
uploaded_filename = list(uploaded.keys())[0]
shutil.move(uploaded_filename, "kaggle.json")

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
shutil.copy("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

!pip install kagglehub -q

import kagglehub
path = kagglehub.dataset_download("vinjamuripavan/bird-species")
print("Path to dataset files:", path)

!ls "{path}"

Saving kaggle (1).json to kaggle (1).json


100%|██████████| 1.97G/1.97G [00:23<00:00, 89.6MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/vinjamuripavan/bird-species/versions/1
'BIRDS 525 SPECIES'


In [3]:
# re-size all the images to this
IMAGE_SIZE = [229, 229]#as per the xception model requirement

In [4]:
train_path = path + '/BIRDS 525 SPECIES/train'
valid_path = path + '/BIRDS 525 SPECIES/valid'
test_path = path + '/BIRDS 525 SPECIES/test'

print(train_path)
print(valid_path)
print(test_path)

/root/.cache/kagglehub/datasets/vinjamuripavan/bird-species/versions/1/BIRDS 525 SPECIES/train
/root/.cache/kagglehub/datasets/vinjamuripavan/bird-species/versions/1/BIRDS 525 SPECIES/valid
/root/.cache/kagglehub/datasets/vinjamuripavan/bird-species/versions/1/BIRDS 525 SPECIES/test


In [5]:
# add preprocessing layer to the front of EfficientNet
xcept = Xception(input_shape=IMAGE_SIZE + [3], weights='imagenet', include_top=False)

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [6]:
# don't train existing weights
for layer in xcept.layers:
  layer.trainable = False

In [7]:
#Getting Number of Categories
folders = glob(train_path + '/*')

In [8]:
x = Flatten()(xcept.output)

In [9]:
x = layers.Dense(256, 'relu', kernel_initializer='he_normal')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)

In [10]:
prediction = Dense(len(folders), activation='softmax')(x)

In [11]:
# create a model object
model = Model(inputs=xcept.input, outputs=prediction)

In [12]:
# view the structure of the model
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 229, 229,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1        │ (None, 114, 114,  │        864 │ input_layer[0][0] │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_bn     │ (None, 114, 114,  │        128 │ block1_conv1[0][… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_act    │ (None, 114, 114,  │          0 │ block1_conv1_bn[… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2        │ (None, 112, 112,  │     18,432 │ block1_conv1_act… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_bn     │ (None, 112, 112,  │        256 │ block1_conv2[0][… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_act    │ (None, 112, 112,  │          0 │ block1_conv2_bn[… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1     │ (None, 112, 112,  │      8,768 │ block1_conv2_act… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1_bn  │ (None, 112, 112,  │        512 │ block2_sepconv1[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_act │ (None, 112, 112,  │          0 │ block2_sepconv1_… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2     │ (None, 112, 112,  │     17,536 │ block2_sepconv2_… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_bn  │ (None, 112, 112,  │        512 │ block2_sepconv2[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 56, 56,    │      8,192 │ block1_conv2_act… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pool         │ (None, 56, 56,    │          0 │ block2_sepconv2_… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 56, 56,    │        512 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 56, 56,    │          0 │ block2_pool[0][0… │
│                     │ 128)              │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_sepconv1_act │ (None, 56, 56,    │          0 │ add[0][0]       

 Total params: 46,687,797 (178.10 MB)

 Trainable params: 25,825,805 (98.52 MB)

 Non-trainable params: 20,861,992 (79.58 MB)

In [13]:
# tell the model what cost and optimization method to use
model.compile(
  loss='categorical_crossentropy',
  optimizer='adam',
  metrics=['accuracy']
)

In [14]:
#Data Augmentation
import tensorflow as tf
from tensorflow.keras import layers

# The deprecated tf.keras.preprocessing.image_data_generator is replaced with tf.keras.utils library as per the latest keras version

IMG_SIZE = 229
BATCH_SIZE=32

training_set = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)
testing_set=tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

valid_set=tf.keras.utils.image_dataset_from_directory(
    valid_path,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical'

)
resize_and_rescale = tf.keras.Sequential([
  layers.Resizing(IMG_SIZE, IMG_SIZE),
  layers.Lambda(lambda x: (x / 127.5) - 1.0)
])
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"), #no vertical flip as that would cause unncessary distortion
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15)
])
training_set = training_set.map(lambda x, y: (data_augmentation(resize_and_rescale(x, training=True), training=True), y))


valid_set = valid_set.map(lambda x, y: (resize_and_rescale(x, training=False), y))
testing_set = testing_set.map(lambda x, y: (resize_and_rescale(x, training=False), y))

Found 84635 files belonging to 525 classes.
Found 2625 files belonging to 525 classes.
Found 2625 files belonging to 525 classes.


In [ ]:
#Fitting the model
r = model.fit(
  training_set,
  validation_data=valid_set,
  epochs=10,
  steps_per_epoch=len(training_set),
  validation_steps=len(valid_set)
)


Epoch 1/10
2645/2645 ━━━━━━━━━━━━━━━━━━━━ 1302s 481ms/step - accuracy: 0.4955 - loss: 2.2525 - val_accuracy: 0.7794 - val_loss: 0.7983
Epoch 2/10
2645/2645 ━━━━━━━━━━━━━━━━━━━━ 1227s 464ms/step - accuracy: 0.6617 - loss: 1.3264 - val_accuracy: 0.8278 - val_loss: 0.6361
Epoch 3/10
2645/2645 ━━━━━━━━━━━━━━━━━━━━ 1216s 460ms/step - accuracy: 0.6955 - loss: 1.1700 - val_accuracy: 0.8526 - val_loss: 0.5247
Epoch 4/10
1907/2645 ━━━━━━━━━━━━━━━━━━━━ 5:40 461ms/step - accuracy: 0.7154 - loss: 1.0933

In [ ]:
#Saving the Model.
import tensorflow as tf

from keras.models import load_model

model.save('./bird_classification_new_model.h5')

/opt/conda/lib/python3.7/site-packages/keras/utils/generic_utils.py:497: CustomMaskWarning: Custom mask layers require a config and must override get_config. When loading, the custom mask layer must be passed to the custom_objects argument.
  category=CustomMaskWarning)


In [ ]:
model.evaluate(test_set)

63/63 [==============================] - 13s 201ms/step - loss: 0.1070 - accuracy: 0.9490
[0.10677755026817322, 0.9489999895095825]
